# TinyCeNN-LM: optimized continuation beyond the first 10M tokens

This recipe continues an existing **Transformer-free** dense CeNN student. It uses FP32 parameter storage with mixed-precision computation, trains the embeddings/final norm/output head at 5% of the core learning rate, and keeps the learning rate steady until the final 20% of the run. Teacher KL gradually decreases from 1.0 to 0.25 and hidden alignment from 0.25 to 0.

The held-out benchmark remains fixed. Both best and final models retain their own weights, metrics, and training-stream positions. Legacy checkpoints have estimated stream offsets; new checkpoints store an exact offset for unchanged data/tokenization/shuffle settings. Continuation starts a new AdamW optimizer and schedule.

**This is a testable training recipe, not a guarantee that validation loss will keep improving indefinitely.** Compare the held-out CE/perplexity and recent plateau indicator. Use a Colab GPU runtime.


In [ ]:
import subprocess, sys, pathlib, importlib
subprocess.run(["nvidia-smi"], check=False)

# This branch contains the optimization changes; use main after the PR is merged.
REPO_REF = "codex/fix-training-plateau"
REPO_DIR = pathlib.Path("/content/TinyCeNN-LM-optimized")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF,
                    "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()


## Hugging Face login

Add a Hugging Face **write token** to Colab Secrets as `HF_TOKEN`.


In [ ]:
from huggingface_hub import login, HfApi, snapshot_download

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()

api = HfApi()
hf_user = api.whoami()["name"]
print("Logged in as:", hf_user)


In [ ]:
# Select the plateaued dense checkpoint. Use .../TinyCeNN-LM-Distilled for the original 10M run.
SOURCE_HF_REPO = f"{hf_user}/TinyCeNN-LM-Distilled-v2"
resume_student = snapshot_download(repo_id=SOURCE_HF_REPO, repo_type="model")
print("Resume checkpoint:", resume_student)


In [ ]:
from datetime import datetime
import json

ADDITIONAL_TOKENS = 30_000_000
CONTEXT_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 3e-4

# More adaptation capacity, without adding Transformer layers.
TRAIN_INTERFACES = "all"    # "none": core only; "norm": core + final norm
INTERFACE_LR_SCALE = 0.05
LR_SCHEDULE = "wsd"        # warmup, stable, decay
DECAY_RATIO = 0.20
KL_FINAL_WEIGHT = 0.25
HIDDEN_FINAL_WEIGHT = 0.0

# Inherit the architecture and benchmark settings of the selected checkpoint.
metadata = json.loads((pathlib.Path(resume_student) / "student_config.json").read_text())
previous_path = pathlib.Path(resume_student) / "distillation_report.json"
previous = json.loads(previous_path.read_text()) if previous_path.exists() else {}
CENN_STEPS = metadata["cenn"]["steps"]
DILATIONS = ",".join(map(str, metadata["cenn"]["dilations"]))
CONTEXT_LENGTH = previous.get("context_length", CONTEXT_LENGTH)
BASE_MODEL = metadata["base_model"]
DATASET = previous.get("dataset", "HuggingFaceFW/fineweb")
DATASET_CONFIG = previous.get("dataset_config", "sample-10BT")
DATASET_REVISION = previous.get("dataset_revision")
DATASET_SPLIT = previous.get("dataset_split", "train")
TEXT_FIELD = previous.get("text_field", "text")
TEMPERATURE = 2.0
CE_WEIGHT = 1.0
KL_WEIGHT = 1.0
HIDDEN_WEIGHT = 0.25
SHUFFLE_BUFFER = previous.get("training_shuffle", {}).get("buffer_size", 4096)
SEED = previous.get("training_shuffle", {}).get("seed", 42)
EVAL_BATCHES = previous.get("evaluation", {}).get("batches", 64)
EVAL_BATCH_SIZE = previous.get("evaluation", {}).get("batch_size", 4)
EVAL_EVERY = 250
TARGET_GAP_RECOVERY = 0.90

# Each execution gets a new output directory; earlier checkpoints stay available.
OUTPUT_DIR = str(REPO_DIR / "checkpoints" / f"cenn-optimized-{datetime.now():%Y%m%d-%H%M%S}")
print("Held-out benchmark tokens:", EVAL_BATCHES * EVAL_BATCH_SIZE * CONTEXT_LENGTH)
print("Output:", OUTPUT_DIR)


In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / "scripts/train_distill_rigorous.py"),
    "--resume-student-dir", resume_student,
    "--base-model", BASE_MODEL,
    "--dataset", DATASET,
    "--dataset-config", DATASET_CONFIG,
    "--split", DATASET_SPLIT,
    "--text-field", TEXT_FIELD,
    "--seed", str(SEED),
    "--kernel-size", str(metadata["cenn"]["kernel_size"]),
    "--expansion", str(metadata["cenn"]["expansion"]),
    "--train-interfaces", TRAIN_INTERFACES,
    "--interface-lr-scale", str(INTERFACE_LR_SCALE),
    "--lr-schedule", LR_SCHEDULE,
    "--decay-ratio", str(DECAY_RATIO),
    "--kl-final-weight", str(KL_FINAL_WEIGHT),
    "--hidden-final-weight", str(HIDDEN_FINAL_WEIGHT),
    "--max-tokens", str(ADDITIONAL_TOKENS),
    "--context-length", str(CONTEXT_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--learning-rate", str(LEARNING_RATE),
    "--steps", str(CENN_STEPS),
    "--dilations", DILATIONS,
    "--temperature", str(TEMPERATURE),
    "--ce-weight", str(CE_WEIGHT),
    "--kl-weight", str(KL_WEIGHT),
    "--hidden-weight", str(HIDDEN_WEIGHT),
    "--shuffle-buffer", str(SHUFFLE_BUFFER),
    "--eval-batches", str(EVAL_BATCHES),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--eval-every", str(EVAL_EVERY),
    "--target-gap-recovery", str(TARGET_GAP_RECOVERY),
    "--output-dir", OUTPUT_DIR,
]
if DATASET_REVISION:
    cmd += ["--dataset-revision", DATASET_REVISION]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
# Inspect the rigorous report.
import json

report_path = pathlib.Path(OUTPUT_DIR) / "distillation_report.json"
report = json.loads(report_path.read_text())

summary = {
    "status": report["status"],
    "plateau": report["plateau"],
    "precision": report["training_precision"],
    "training_stream": report["training_stream"],
    "train_interfaces": report["train_interfaces"],
    "benchmark_protocol": report["benchmark_protocol"],
    "held_out_tokens": report["evaluation"]["tokens"],
    "evaluation_fingerprint": report["evaluation"]["fingerprint_sha256"],
    "previous_training_tokens": report["previous_training_tokens"],
    "tokens_this_run": report["seen_tokens_this_run"],
    "cumulative_training_tokens": report["cumulative_training_tokens"],
    "cold_student_ce": report["cold_initial"]["student_ce"],
    "run_start_ce": report["run_start"]["student_ce"],
    "best_student_ce": report["best"]["student_ce"],
    "teacher_ce": report["best"]["teacher_ce"],
    "best_student_ppl": report["best"]["student_ppl"],
    "teacher_ppl": report["best"]["teacher_ppl"],
    "teacher_gap_recovery_percent": 100 * report["teacher_gap_recovery_fraction"],
    "target_reached": report["target_gap_recovery_reached"],
}
print(json.dumps(summary, indent=2))

if report["status"] == "diverged":
    raise RuntimeError("Continuation distillation diverged.")


In [ ]:
# Select the rigorous best checkpoint.
best_dir = pathlib.Path(OUTPUT_DIR + "-best")
publish_dir = best_dir if best_dir.exists() else pathlib.Path(OUTPUT_DIR)
print("Selected checkpoint:", publish_dir)
report = json.loads((publish_dir / "distillation_report.json").read_text())
checkpoint_tokens = report["checkpoint"]["cumulative_training_tokens"]

card = f'''---
base_model: arnir0/Tiny-LLM
library_name: transformers
pipeline_tag: text-generation
tags:
- cenn
- knowledge-distillation
- transformer-free
- language-modeling
- recurrent-neural-network
- rigorous-benchmark
---

# TinyCeNN-LM Optimized Continuation

Transformer-free CeNN student distilled from `arnir0/Tiny-LLM`.

## Architecture
- Transformer layers remaining: 0
- CeNN recurrent steps: {report['cenn_steps']}
- CeNN receptive field: {report['cenn_receptive_field']} tokens
- Trainable parameters (core + selected interfaces): {report['parameters']['trainable']:,}

## Rigorous benchmark
- Protocol: {report['benchmark_protocol']}
- Deterministic held-out tokens: {report['evaluation']['tokens']:,}
- Benchmark SHA256: `{report['evaluation']['fingerprint_sha256']}`
- Cumulative distillation tokens: {checkpoint_tokens:,}
- Best student CE: {report['best']['student_ce']:.6f}
- Teacher CE: {report['best']['teacher_ce']:.6f}
- Best student PPL: {report['best']['student_ppl']:.3f}
- Teacher PPL: {report['best']['teacher_ppl']:.3f}
- Teacher-gap recovery: {100*report['teacher_gap_recovery_fraction']:.2f}%

Load via `tinycenn_lm.build_cenn_student()`.
'''
(publish_dir / "README.md").write_text(card, encoding="utf-8")


In [ ]:
# Publish as a versioned model so the original 10M checkpoint remains reproducible.
HF_MODEL_NAME = "TinyCeNN-LM-Distilled-v3"
HF_REPO_ID = f"{hf_user}/{HF_MODEL_NAME}"

api.create_repo(HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(publish_dir),
    commit_message=(
        f"Optimized CeNN continuation: "
        f"{checkpoint_tokens:,} cumulative tokens, "
        f"{100*report['teacher_gap_recovery_fraction']:.2f}% gap recovery"
    ),
)
print(f"https://huggingface.co/{HF_REPO_ID}")


In [ ]:
# Re-download the published artifact and reproduce the exact held-out benchmark.
parity_cmd = [
    sys.executable,
    str(REPO_DIR / "scripts/eval_distilled.py"),
    "--hf-repo", HF_REPO_ID,
    "--ce-tolerance", "0.02",
]
subprocess.run(parity_cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
# Side-by-side deterministic generation: teacher vs remotely reloaded CeNN student.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm import build_cenn_student, CeNNReplacementLayer

downloaded_v2 = snapshot_download(repo_id=HF_REPO_ID, repo_type="model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == "cuda" else torch.float32)
)

tokenizer = AutoTokenizer.from_pretrained(downloaded_v2)
student = build_cenn_student(downloaded_v2, device=device, dtype=torch.float32).eval()
teacher = AutoModelForCausalLM.from_pretrained(
    "arnir0/Tiny-LLM",
    dtype=dtype if device.type == "cuda" else None,
    attn_implementation="sdpa",
).to(device).eval()

assert len([m for m in student.modules() if isinstance(m, CeNNReplacementLayer)]) == 1
assert not any("self_attn" in name or "mlp" in name for name, _ in student.named_modules())
print("Transformer-free structure: PASS")

prompts = [
    "The capital of Austria is",
    "Artificial intelligence can help",
    "A small language model",
    "In the future, efficient AI",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    generation_args = dict(
        max_new_tokens=40,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    with torch.inference_mode():
        student_out = student.generate(**inputs, **generation_args)
        teacher_out = teacher.generate(**inputs, **generation_args)

    print("\nPROMPT:", prompt)
    print("TEACHER:", tokenizer.decode(teacher_out[0], skip_special_tokens=True))
    print("STUDENT:", tokenizer.decode(student_out[0], skip_special_tokens=True))
